# Practical 4: Prompt Engineering and Retrieval Augmented Generation (RAG)

## Aim
To design effective prompts and implement Retrieval Augmented Generation to reduce hallucination in large language models.

In [ ]:
%pip install -q transformers scikit-learn sentencepiece

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
import numpy as np

print('Libraries loaded successfully!')

## Step 1: Build a Small Knowledge Base

In [ ]:
documents = [
    'RAG combines retrieval and generation to improve factual correctness.',
    'Prompt engineering includes role prompting, constraints, and structured output formats.',
    'Hallucination is when an LLM generates false or unsupported information.',
    'A retriever fetches relevant context chunks from a knowledge base.',
    'The generator uses retrieved context to produce grounded answers.'
]

vectorizer = TfidfVectorizer()
doc_matrix = vectorizer.fit_transform(documents)

print('Knowledge base indexed with TF-IDF.')

In [ ]:
def retrieve_context(query, top_k=2):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, doc_matrix).flatten()
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [documents[i] for i in top_idx]

print('Retriever ready.')

## Step 2: Load a Small Instruction Model

In [ ]:
generator = pipeline('text2text-generation', model='google/flan-t5-small')
print('FLAN-T5 model loaded.')

## Step 3: Baseline Prompt vs RAG Prompt

In [ ]:
def baseline_answer(question):
    prompt = f'Answer the question in 2-3 lines. Question: {question}'
    return generator(prompt, max_new_tokens=80)[0]['generated_text']

def rag_answer(question):
    context = retrieve_context(question, top_k=2)
    context_text = '\n'.join(context)
    prompt = (
        'You are a factual assistant. Use ONLY the context below. '
        'If answer is not in context, say: Not available in the provided context.\n\n'
        f'Context:\n{context_text}\n\nQuestion: {question}\nAnswer:'
    )
    return generator(prompt, max_new_tokens=90)[0]['generated_text'], context

question = 'How does RAG reduce hallucination in LLMs?'

base_out = baseline_answer(question)
rag_out, retrieved = rag_answer(question)

print('Question:', question)
print('\nRetrieved Context:')
for i, ctx in enumerate(retrieved, start=1):
    print(f'{i}. {ctx}')

print('\nBaseline Output:')
print(base_out)

print('\nRAG Output:')
print(rag_out)

## Observation
- Baseline prompting may produce generic or unsupported claims.
- RAG prompting grounds the answer in retrieved context.
- Constraint-based prompts reduce hallucination by forcing evidence-aware generation.

## Result
Prompt engineering and a simple RAG pipeline were implemented successfully to improve factual consistency in LLM responses.